# Experimento de ETL - Integração CSV para SQL

Extração, transformação e carga dos nomes de usuários entre as tabelas MySQL.

In [ ]:
import sys
from pathlib import Path

raiz = Path.cwd()
if not (raiz / 'src').is_dir():
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

import pandas as pd
from sqlalchemy import text
from src.database.connection import get_engine
from src.components.transformers import processar_usuarios_etl
from src.main import validar_esquema

engine = get_engine()
with engine.connect() as conexao:
    conexao.execute(text('SELECT 1'))
print('Conexão estabelecida!')

In [ ]:
# Extração (Extract)
df_bruto = pd.read_sql(text('SELECT * FROM usuarios_temporarios'), engine)
print(f'{len(df_bruto)} registros extraídos.')
df_bruto.head()

In [ ]:
# Transformação (Transform)
df_transformado = processar_usuarios_etl(df_bruto)
print('Dados transformados e padronizados:')
df_transformado.head()

In [ ]:
# Carga (Load)
validar_esquema(df_transformado, engine)
df_transformado.to_sql(
    'usuarios_final', con=engine, if_exists='append', index=False
)
print("Carga finalizada com sucesso na tabela 'usuarios_final'!")